# Per-2-Hour-Epoch Sampled Scatter Decoding Orchestrator — Quality-Filtered Speech

Combines the 2-hour-epoch structure of `scat_classifier_sampled_nocv_filtered_speech_per_day.ipynb`
with fine-grained temporal splitting.

Epoch blocks: 09-11, 11-13, 13-15, 15-17, 17-19, 19-21, 21-23 **Central local time**.  
One SLURM job is submitted per *(patient, date, epoch, resample)* tuple.  
Outputs land in `{vad_root}/{patient}/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_per_epoch/{date}/{epoch}/`.

**`REUSE_DAY_CONSENSUS = True`** (default): borrows hyperparameters from the already-finished
per-day run (`scat_xgboost_sampled_norm_filtered_per_day/{date}/`) for the same *(patient, date)*,
going straight to the 20-resample fixed-params stage.  
Set to `False` to run the full two-phase search independently for each epoch.

### 1. Imports

In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────────────────
PATIENTS    = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG',
               'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

BASE_RUN_NAME = 'scat_xgboost_sampled_norm_filtered_per_epoch'
# Source of borrowed consensus — the per-day run for the same (patient, date)
DAY_RUN_NAME  = 'scat_xgboost_sampled_norm_filtered_per_day'

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE      = {}

# ── Epoch settings ────────────────────────────────────────────────────────────────────────
# 2-hour blocks in Central local time (inclusive start, exclusive end)
LOCAL_TZ    = 'America/Chicago'
HOUR_BLOCKS = [(9, 11), (11, 13), (13, 15), (15, 17), (17, 19), (19, 21), (21, 23)]
MIN_WORDS_PER_EPOCH = 200   # silently skip epochs below this threshold

# ── Two-phase strategy ────────────────────────────────────────────────────────────────────
# True  → skip per-epoch hyperparam search; borrow consensus from the per-day run
# False → run full 3+17 pipeline independently per epoch
REUSE_DAY_CONSENSUS = True

N_RESAMPLES  = 20
FIRST_STAGE  = [0, 1, 2]
SECOND_STAGE = list(range(3, N_RESAMPLES))
SEED_STRIDE  = 42

PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_decoding_analysis/scat_classifier_worker.py')

PARTITION = 'Universal'
QOS       = 'big_batch_tier'
CPUS      = 8
MEM       = '40G'
TIME      = '48:00:00'

TEST_SIZE    = 0.2
SEARCH_ITERS = 40
CV_SPLITS    = 4
N_SHUFFLES   = 50
SCORING      = 'f1_macro'
PERM_METRIC  = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)
print(f'local tz:    {LOCAL_TZ}')
print(f'hour blocks: {HOUR_BLOCKS}')
print(f'reuse day consensus: {REUSE_DAY_CONSENSUS}')

patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py
local tz:    America/Chicago
hour blocks: [(9, 11), (11, 13), (13, 15), (15, 17), (17, 19), (19, 21), (21, 23)]
reuse day consensus: True


### 3. Compute & Save Per-Epoch Word Indices Per Patient

In [3]:
def epoch_str(start: int, end: int) -> str:
    return f'{start:02d}-{end:02d}'


def compute_epoch_indices(patient: str) -> 'dict[tuple, Path]':
    patient_root = VAD_ROOT / patient
    filtered_csv = patient_root / f'{patient}_word_df_filtered.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}

    df = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    df['epoch'] = None
    for start, end in HOUR_BLOCKS:
        mask = (df['hour'] >= start) & (df['hour'] < end)
        df.loc[mask, 'epoch'] = epoch_str(start, end)
    df = df[df['epoch'].notna()].copy()

    epoch_paths = {}
    for (date_str, ep), grp in df.groupby(['date_str', 'epoch']):
        if len(grp) < MIN_WORDS_PER_EPOCH:
            continue  # silently skip
        idx_dir  = patient_root / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str / ep
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_{ep}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        epoch_paths[(date_str, ep)] = idx_path
        print(f'  [{patient}] {date_str} {ep}: {len(word_idx):,} words → {idx_path.name}')

    return epoch_paths


print(f'Computing per-epoch word indices for {len(PATIENTS)} patients...\n')
patient_epoch_idx_paths = {}
for patient in PATIENTS:
    ep_paths = compute_epoch_indices(patient)
    if ep_paths:
        patient_epoch_idx_paths[patient] = ep_paths

total_triplets = sum(len(v) for v in patient_epoch_idx_paths.values())
print(f'\nTotal (patient, date, epoch) triplets ready: {total_triplets}')

Computing per-epoch word indices for 20 patients...

  [YEY] 2024-04-01 09-11: 1,116 words → YEY_2024-04-01_09-11_word_idx.npy
  [YEY] 2024-04-01 11-13: 3,378 words → YEY_2024-04-01_11-13_word_idx.npy
  [YEY] 2024-04-01 13-15: 4,508 words → YEY_2024-04-01_13-15_word_idx.npy
  [YEY] 2024-04-01 15-17: 7,508 words → YEY_2024-04-01_15-17_word_idx.npy
  [YEY] 2024-04-02 11-13: 5,976 words → YEY_2024-04-02_11-13_word_idx.npy
  [YEY] 2024-04-02 13-15: 2,716 words → YEY_2024-04-02_13-15_word_idx.npy
  [YEY] 2024-04-02 15-17: 11,636 words → YEY_2024-04-02_15-17_word_idx.npy
  [YEY] 2024-04-02 17-19: 3,513 words → YEY_2024-04-02_17-19_word_idx.npy
  [YEZ] 2024-04-09 15-17: 652 words → YEZ_2024-04-09_15-17_word_idx.npy
  [YEZ] 2024-04-09 17-19: 2,600 words → YEZ_2024-04-09_17-19_word_idx.npy
  [YEZ] 2024-04-09 19-21: 858 words → YEZ_2024-04-09_19-21_word_idx.npy
  [YEZ] 2024-04-09 21-23: 2,588 words → YEZ_2024-04-09_21-23_word_idx.npy
  [YEZ] 2024-04-10 09-11: 3,704 words → YEZ_2024-04-10_09-11_w

### 4. Path Resolution & Consensus-Params Helpers

In [4]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_neural_inputs(patient: str) -> dict:
    root = VAD_ROOT / patient
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override     = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path           = first_existing([frs_override, new_frs, legacy_frs])
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path           = new_frs
    else:
        cluster_preds_path = None
        frs_path           = None

    return {'cluster_preds_path': cluster_preds_path, 'frs_path': frs_path}


def day_consensus_path(patient: str, date_str: str) -> 'Path | None':
    """Return existing consensus params from the per-day run for this (patient, date)."""
    p = VAD_ROOT / patient / 'standard_decoding_analysis' / DAY_RUN_NAME / date_str / 'consensus_best_params.json'
    return p if p.exists() else None


def choose_consensus_params(param_options: list) -> dict:
    params = dict(param_options[0])
    params['max_depth']        = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma']            = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda']       = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha']        = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample']        = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate']    = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators']     = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params

### 5. Build Status Table

In [5]:
def resample_paths(out_root: Path, r: int) -> dict:
    return {
        'pkl':              out_root / f'scat_sampled_resample_{r}_.pkl',
        'summary_json':     out_root / f'summary_resample_{r}.json',
        'best_params_json': out_root / f'best_params_resample_{r}.json',
        'success':          out_root / f'resample_{r}_SUCCESS',
        'error':            out_root / f'resample_{r}_error.txt',
    }


def build_status_df() -> pd.DataFrame:
    rows = []
    for patient in PATIENTS:
        neural = resolve_patient_neural_inputs(patient)
        ep_idx_map = patient_epoch_idx_paths.get(patient, {})

        for (date_str, ep), word_idx_path in ep_idx_map.items():
            out_root = VAD_ROOT / patient / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str / ep
            out_root.mkdir(parents=True, exist_ok=True)
            consensus_json = out_root / 'consensus_best_params.json'
            has_inputs = (neural['cluster_preds_path'] is not None
                         and neural['frs_path'] is not None)

            # When reusing day consensus, copy from the per-day run if not there yet
            if REUSE_DAY_CONSENSUS and not consensus_json.exists():
                borrowed = day_consensus_path(patient, date_str)
                if borrowed is not None:
                    consensus_json.write_text(borrowed.read_text())

            consensus_ready = consensus_json.exists()

            if REUSE_DAY_CONSENSUS:
                effective_first_stage  = []
                effective_second_stage = list(range(N_RESAMPLES))
            else:
                effective_first_stage  = FIRST_STAGE
                effective_second_stage = SECOND_STAGE

            first_three_done = all(
                (out_root / f'resample_{r}_SUCCESS').exists()
                for r in effective_first_stage
            ) if effective_first_stage else True

            for r in range(N_RESAMPLES):
                paths = resample_paths(out_root, r)
                rows.append({
                    'patient':             patient,
                    'date':                date_str,
                    'epoch':               ep,
                    'resample_idx':        r,
                    'stage':               'hyperparam' if r in effective_first_stage else 'fixed',
                    'seed':                r * SEED_STRIDE,
                    'cluster_preds_path':  neural['cluster_preds_path'],
                    'frs_path':            neural['frs_path'],
                    'word_idx_path':       word_idx_path,
                    'has_inputs':          has_inputs,
                    'out_root':            out_root,
                    'consensus_json':      consensus_json,
                    'consensus_ready':     consensus_ready,
                    'first_three_done':    first_three_done,
                    'pkl_path':            paths['pkl'],
                    'summary_json':        paths['summary_json'],
                    'best_params_json':    paths['best_params_json'],
                    'success_path':        paths['success'],
                    'error_path':          paths['error'],
                    'has_pkl':             paths['pkl'].exists(),
                    'has_success':         paths['success'].exists(),
                    'has_error':           paths['error'].exists(),
                    'ready_phase1':        (has_inputs and r in effective_first_stage
                                          and not paths['success'].exists()),
                    'ready_phase2':        (has_inputs and r in effective_second_stage
                                          and first_three_done and consensus_ready
                                          and not paths['success'].exists()),
                })
    return pd.DataFrame(rows).sort_values(['patient', 'date', 'epoch', 'resample_idx']).reset_index(drop=True)


status_df = build_status_df()
print(f'total rows (patient × date × epoch × resample): {len(status_df)}')
print(f'has inputs:      {int(status_df["has_inputs"].sum())}')
print(f'consensus ready: {int(status_df["consensus_ready"].sum())}')
print(f'completed:       {int(status_df["has_success"].sum())}')
print(f'errors:          {int(status_df["has_error"].sum())}')
print(f'ready_phase1:    {int(status_df["ready_phase1"].sum())}')
print(f'ready_phase2:    {int(status_df["ready_phase2"].sum())}')
status_df[['patient','date','epoch','resample_idx','stage','has_inputs','consensus_ready',
           'has_success','has_error','ready_phase1','ready_phase2']].head(50)

total rows (patient × date × epoch × resample): 16280
has inputs:      16280
consensus ready: 16280
completed:       0
errors:          0
ready_phase1:    0
ready_phase2:    16280


,patient,date,epoch,resample_idx,stage,has_inputs,consensus_ready,has_success,has_error,ready_phase1,ready_phase2
0,YEY,2024-04-01,09-11,0,fixed,True,True,False,False,False,True
1,YEY,2024-04-01,09-11,1,fixed,True,True,False,False,False,True
2,YEY,2024-04-01,09-11,2,fixed,True,True,False,False,False,True
3,YEY,2024-04-01,09-11,3,fixed,True,True,False,False,False,True
4,YEY,2024-04-01,09-11,4,fixed,True,True,False,False,False,True
5,YEY,2024-04-01,09-11,5,fixed,True,True,False,False,False,True
6,YEY,2024-04-01,09-11,6,fixed,True,True,False,False,False,True
7,YEY,2024-04-01,09-11,7,fixed,True,True,False,False,False,True
8,YEY,2024-04-01,09-11,8,fixed,True,True,False,False,False,True
9,YEY,2024-04-01,09-11,9,fixed,True,True,False,False,False,True


### 6. (Patient, Date, Epoch) Input Summary

In [6]:
triplet_df = (
    status_df.drop_duplicates(subset=['patient', 'date', 'epoch'])
    [['patient', 'date', 'epoch', 'has_inputs', 'consensus_ready', 'cluster_preds_path', 'frs_path', 'word_idx_path']]
    .sort_values(['patient', 'date', 'epoch'])
    .reset_index(drop=True)
)
print(f'{len(triplet_df)} (patient, date, epoch) triplets')
display(triplet_df)

814 (patient, date, epoch) triplets


,patient,date,epoch,has_inputs,consensus_ready,cluster_preds_path,frs_path,word_idx_path
0,YEY,2024-04-01,09-11,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YEY,2024-04-01,11-13,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YEY,2024-04-01,13-15,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YEY,2024-04-01,15-17,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YEY,2024-04-02,11-13,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
...,...,...,...,...,...,...,...,...
809,YFV,2026-02-20,09-11,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
810,YFV,2026-02-20,11-13,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
811,YFV,2026-02-20,13-15,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
812,YFV,2026-02-20,15-17,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


### 7a. Submit Phase-1 Jobs  
*(Only runs if `REUSE_DAY_CONSENSUS = False`; otherwise `ready_phase1` is always empty.)*

In [7]:
DRY_RUN = False

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase1']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    ep       = row['epoch']
    r        = int(row['resample_idx'])
    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=scat_fse_{patient}_{date_str}_{ep}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{date_str}_{ep}_r{r}_%j.out
#SBATCH --error={logs_dir}/{patient}_{date_str}_{ep}_r{r}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "DATE: {date_str}"
echo "EPOCH: {ep}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --resample-idx {r} \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --out-dir "{row['out_root']}" \\
  {word_idx_arg} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --seed-stride {SEED_STRIDE} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{date_str}_{ep}_resample_{r}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, ep, r, 'dry-run'))
        print(f'dry-run: {patient} {date_str} {ep} r={r}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, ep, r, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} {ep} r={r} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, ep, r, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} {ep} r={r}\n{exc.stderr}')

print(f'phase-1 submitted={len(submitted)}, failed={len(failed)}')

phase-1 submitted=0, failed=0


### 7b. Build Per-Epoch Consensus Params  
*(Only needed if `REUSE_DAY_CONSENSUS = False`; otherwise consensus was copied in the status step.)*

In [8]:
if not REUSE_DAY_CONSENSUS:
    built   = []
    blocked = []
    for (patient, date_str, ep), grp in status_df.groupby(['patient', 'date', 'epoch']):
        out_root       = grp.iloc[0]['out_root']
        consensus_json = grp.iloc[0]['consensus_json']
        if consensus_json.exists():
            print(f'consensus already exists: {patient} {date_str} {ep}')
            continue
        first_three = grp[grp['resample_idx'].isin(FIRST_STAGE)]
        if not bool(first_three['has_success'].all()):
            blocked.append((patient, date_str, ep))
            continue
        param_options = []
        for r in FIRST_STAGE:
            path = out_root / f'best_params_resample_{r}.json'
            param_options.append(json.loads(path.read_text()))
        consensus = choose_consensus_params(param_options)
        consensus_json.write_text(json.dumps(consensus, indent=2))
        built.append((patient, date_str, ep))
        print(f'built consensus: {patient} {date_str} {ep}')
    print('built:', len(built), '  blocked:', len(blocked))
else:
    print('REUSE_DAY_CONSENSUS=True — consensus already copied during status build.')

REUSE_DAY_CONSENSUS=True — consensus already copied during status build.


### 7c. Submit Phase-2 (Fixed-Params) Jobs

In [9]:
DRY_RUN = False

# Refresh status after any consensus files were written
status_df = build_status_df()

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase2']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    ep       = row['epoch']
    r        = int(row['resample_idx'])
    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=scat_fse_{patient}_{date_str}_{ep}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{date_str}_{ep}_r{r}_%j.out
#SBATCH --error={logs_dir}/{patient}_{date_str}_{ep}_r{r}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "DATE: {date_str}"
echo "EPOCH: {ep}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --resample-idx {r} \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --out-dir "{row['out_root']}" \\
  --params-json "{row['consensus_json']}" \\
  {word_idx_arg} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --seed-stride {SEED_STRIDE} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{date_str}_{ep}_resample_{r}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, ep, r, 'dry-run'))
        print(f'dry-run: {patient} {date_str} {ep} r={r}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, ep, r, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} {ep} r={r} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, ep, r, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} {ep} r={r}\n{exc.stderr}')

print(f'phase-2 submitted={len(submitted)}, failed={len(failed)}')

submitted: YEY 2024-04-01 09-11 r=0 -> Submitted batch job 274590
submitted: YEY 2024-04-01 09-11 r=1 -> Submitted batch job 274591
submitted: YEY 2024-04-01 09-11 r=2 -> Submitted batch job 274592
submitted: YEY 2024-04-01 09-11 r=3 -> Submitted batch job 274593
submitted: YEY 2024-04-01 09-11 r=4 -> Submitted batch job 274594
submitted: YEY 2024-04-01 09-11 r=5 -> Submitted batch job 274595
submitted: YEY 2024-04-01 09-11 r=6 -> Submitted batch job 274596
submitted: YEY 2024-04-01 09-11 r=7 -> Submitted batch job 274597
submitted: YEY 2024-04-01 09-11 r=8 -> Submitted batch job 274598
submitted: YEY 2024-04-01 09-11 r=9 -> Submitted batch job 274599
submitted: YEY 2024-04-01 09-11 r=10 -> Submitted batch job 274600
submitted: YEY 2024-04-01 09-11 r=11 -> Submitted batch job 274601
submitted: YEY 2024-04-01 09-11 r=12 -> Submitted batch job 274602
submitted: YEY 2024-04-01 09-11 r=13 -> Submitted batch job 274603
submitted: YEY 2024-04-01 09-11 r=14 -> Submitted batch job 274604
submi

### 8. Check Status

In [ ]:
status_df = build_status_df()
triplet_progress = (
    status_df.groupby(['patient', 'date', 'epoch'])
    .agg(
        total=('resample_idx', 'count'),
        done=('has_success', 'sum'),
        errors=('has_error', 'sum'),
    )
    .reset_index()
)
triplet_progress['pct'] = (triplet_progress['done'] / triplet_progress['total'] * 100).round(0).astype(int)
print(f'Overall: {int(status_df["has_success"].sum())} / {len(status_df)} resamples done')
display(triplet_progress)

### 9. Inspect Errors

In [ ]:
error_rows = status_df[status_df['has_error']].copy()
print(f'resamples with error files: {len(error_rows)}')
for _, row in error_rows.head(10).iterrows():
    print('=' * 100)
    print(row['patient'], row['date'], row['epoch'], f'r={row["resample_idx"]}')
    print(row['error_path'].read_text()[:4000])

### 10. Aggregate Results Per (Patient, Date, Epoch)

In [ ]:
for (patient, date_str, ep), grp in status_df.groupby(['patient', 'date', 'epoch']):
    if not bool(grp['has_success'].all()):
        print(f'skipping {patient} {date_str} {ep}: not all resamples finished')
        continue

    out_root     = grp.iloc[0]['out_root']
    summary_rows = []
    aggregate    = {}
    for r in range(N_RESAMPLES):
        with open(out_root / f'summary_resample_{r}.json') as f:
            summary_rows.append(json.load(f))
        with open(out_root / f'scat_sampled_resample_{r}_.pkl', 'rb') as f:
            aggregate[f'resample_{r}'] = pickle.load(f)

    summary_df = pd.DataFrame(summary_rows).sort_values('resample_idx').reset_index(drop=True)
    agg_csv = out_root / 'scat_sampled_all_resamples.csv'
    agg_pkl = out_root / 'scat_sampled_all_resamples.pkl'
    summary_df.to_csv(agg_csv, index=False)
    with open(agg_pkl, 'wb') as f:
        pickle.dump(aggregate, f)
    print(f'aggregated {patient} {date_str} {ep} -> {agg_csv}')
    display(summary_df)